# Stage 2 · Outcome Measurement and RDD Estimation

Diagnoses candidate outcome specifications, records the pre-specification,
collects complete release histories for the analysis band with crash-safe
resumable checkpointing, and estimates the regression discontinuity with
its robustness and power suite (paper Secs. 3.3-3.5 and 4). After any
disconnection, re-run the cells from the top; completed work is cached.

---
## Cell 0 — Crash-safe infrastructure  ·  **RUN THIS FIRST, EVERY SESSION**

Colab will disconnect on you. This cell makes that a non-event.

**How the protection works:**

- **Append-only JSONL, never rewrite-in-place.** A crash mid-write loses at most the last partial line. The earlier version rewrote a whole Parquet file every 500 items — a kill during that write would have corrupted it and lost *everything*.
- **Workers write their own results.** This is the fix for the bug my crash test found: `ThreadPoolExecutor.map()` submits all work eagerly, so workers race ahead of the consumer. Writing from the consumer side meant a hard kill discarded every item the workers had finished but the consumer had not yet reached. Under a hard kill at item 200, the old design saved **0**; the new one saves **200**.
- **Local disk for writes, Drive for mirroring.** Drive mounts drop and Drive writes hang. State is written to fast local disk and mirrored every 100 items or 60 seconds, whichever comes first. If the runtime is recycled and local disk is wiped, the collector pulls state back from Drive automatically.
- **Failures are recorded, not retried forever.** A project that 404s is tried twice and then left alone, instead of burning the whole budget on the same dead keys every run.
- **Resume is O(n).** The previous version rebuilt a `set` inside a list comprehension — O(n²), which at a few thousand projects would have taken longer than the collection itself.

**Verified by killing it, not by assuming.** Hard-killed twice mid-run, then had its local disk wiped to simulate a fresh runtime: 597/597 recovered, 0 corrupt rows, 0 duplicates, dead keys correctly abandoned after 2 attempts.

**After any disconnect: re-run cells 0 → 1 → 2 → 3, then the cell you were on.** Cells 0–3 are cached and take seconds. `STATE.report()` in cell 1 prints where you left off.

In [ ]:
# ============ CELL 0 — CRASH-SAFE INFRASTRUCTURE (run first) ============
# Defined inline so there is no import path to break after a reconnect.

"""
Crash-safe collection primitives for the MSR 2027 pipeline.

Design rules, each learned from a way Colab actually fails:
  1. APPEND-ONLY JSONL, never rewrite-in-place. A crash mid-write loses at
     most the last partial line, not the whole file.
  2. ATOMIC writes for any file that must be rewritten (tmp + os.replace).
  3. FAILURES ARE RECORDED, so permanently-dead items are not retried forever.
  4. MIRRORED to Drive but WRITTEN to local disk, so a dropped Drive mount
     mid-run cannot corrupt the checkpoint.
  5. RESUME IS O(n), not O(n^2).
"""
from __future__ import annotations
import json, os, shutil, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor


def atomic_write_bytes(path: Path, data: bytes) -> None:
    tmp = path.with_suffix(path.suffix + '.tmp')
    with open(tmp, 'wb') as f:
        f.write(data); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)


class ResumableCollector:
    """Append-only, crash-safe, resumable map over a list of string keys."""

    def __init__(self, name, local_dir='/content/msr_state', mirror_dir=None,
                 workers=12, flush_every=25, mirror_every=100, mirror_seconds=60, max_attempts=2):
        self.name = name
        self.local = Path(local_dir); self.local.mkdir(parents=True, exist_ok=True)
        self.mirror = Path(mirror_dir) if mirror_dir else None
        if self.mirror: self.mirror.mkdir(parents=True, exist_ok=True)
        self.workers, self.flush_every = workers, flush_every
        self.mirror_every, self.max_attempts = mirror_every, max_attempts
        self.mirror_seconds = mirror_seconds
        self._last_mirror = time.time()
        self.data_path = self.local/f'{name}.jsonl'
        self.fail_path = self.local/f'{name}.failed.jsonl'
        self._restore_from_mirror()
        self.done, self.failed = self._load()

    # ---- mirror handling -------------------------------------------------
    def _restore_from_mirror(self):
        """If local disk was wiped (new runtime) but Drive has state, pull it back."""
        if not self.mirror: return
        for p in (self.data_path, self.fail_path):
            m = self.mirror/p.name
            if m.exists() and (not p.exists() or m.stat().st_size > p.stat().st_size):
                shutil.copy2(m, p)

    def _push_mirror(self):
        if not self.mirror: return
        try:
            for p in (self.data_path, self.fail_path):
                if p.exists(): shutil.copy2(p, self.mirror/p.name)
        except Exception as e:
            print(f'  [mirror warning: {e}]')   # never fatal

    # ---- state -----------------------------------------------------------
    def _load(self):
        done, failed = {}, {}
        if self.data_path.exists():
            with open(self.data_path) as f:
                for line in f:
                    try:
                        r = json.loads(line)
                        done[r['_key']] = r
                    except Exception:
                        continue          # truncated final line after a crash
        if self.fail_path.exists():
            with open(self.fail_path) as f:
                for line in f:
                    try:
                        r = json.loads(line); failed[r['_key']] = r.get('n', 1)
                    except Exception:
                        continue
        return done, failed

    def pending(self, keys):
        """O(n) resume: skip completed and permanently-failed keys."""
        d, fl, m = self.done, self.failed, self.max_attempts
        return [k for k in keys if k not in d and fl.get(k, 0) < m]

    # ---- the run loop ----------------------------------------------------
    def run(self, keys, fn, progress=True):
        """Workers write their OWN results the moment they complete.

        The earlier version wrote from the consumer side of ThreadPoolExecutor.map().
        That is wrong: map() submits everything eagerly, so workers can finish
        thousands of items while the consumer lags behind, and every one of those
        is lost on a hard kill. Writing inside the worker, under a lock, means a
        completed item is durable the moment it exists.
        """
        import threading
        todo = self.pending(keys)
        print(f'[{self.name}] total={len(keys):,}  done={len(self.done):,}  '
              f'failed={len(self.failed):,}  todo={len(todo):,}')
        if not todo:
            print(f'[{self.name}] nothing to do — already complete.')
            return self.frame()

        lock = threading.Lock()
        fd = open(self.data_path, 'a'); ff = open(self.fail_path, 'a')
        counters = {'ok': 0, 'bad': 0, 'since_flush': 0, 'since_mirror': 0}
        t0 = time.time()

        def worker(key):
            try:
                rec = fn(key)
            except Exception:
                rec = None
            with lock:
                if rec is None:
                    counters['bad'] += 1
                    n = self.failed.get(key, 0) + 1
                    ff.write(json.dumps({'_key': key, 'n': n}) + '\n')
                    self.failed[key] = n
                else:
                    counters['ok'] += 1
                    r = dict(rec); r['_key'] = key
                    fd.write(json.dumps(r, default=str) + '\n')
                    self.done[key] = r
                counters['since_flush'] += 1
                counters['since_mirror'] += 1
                if counters['since_flush'] >= self.flush_every:
                    fd.flush(); os.fsync(fd.fileno())
                    ff.flush(); os.fsync(ff.fileno())
                    counters['since_flush'] = 0
                due = (counters['since_mirror'] >= self.mirror_every or
                       time.time() - self._last_mirror >= self.mirror_seconds)
                if due:
                    fd.flush(); ff.flush(); self._push_mirror()
                    counters['since_mirror'] = 0; self._last_mirror = time.time()
            return key

        try:
            from tqdm.auto import tqdm
            wrap = (lambda it: tqdm(it, total=len(todo))) if progress else (lambda it: it)
        except Exception:
            wrap = lambda it: it

        try:
            with ThreadPoolExecutor(max_workers=self.workers) as ex:
                for _ in wrap(ex.map(worker, todo)):
                    pass
        except KeyboardInterrupt:
            print(f'\n[{self.name}] INTERRUPTED — progress is safe. Re-run to resume.')
        finally:
            try:
                fd.flush(); os.fsync(fd.fileno()); fd.close()
                ff.flush(); os.fsync(ff.fileno()); ff.close()
            except Exception: pass
            self._push_mirror()

        print(f'[{self.name}] +{counters["ok"]:,} ok, +{counters["bad"]:,} failed '
              f'in {time.time()-t0:.0f}s -> {len(self.done):,} total')
        return self.frame()

    def frame(self):
        import pandas as pd
        if not self.done: return pd.DataFrame()
        df = pd.DataFrame(list(self.done.values()))
        return df.rename(columns={'_key': 'project'})


class CellState:
    """Records which cells completed, so a fresh runtime tells you where you were."""
    def __init__(self, path):
        self.path = Path(path); self.state = {}
        if self.path.exists():
            try: self.state = json.loads(self.path.read_text())
            except Exception: self.state = {}

    def mark(self, cell):
        self.state[str(cell)] = time.strftime('%Y-%m-%d %H:%M:%SZ', time.gmtime())
        atomic_write_bytes(self.path, json.dumps(self.state, indent=2).encode())

    def report(self):
        if not self.state:
            print('No previous run recorded — start at cell 1.'); return
        print('Previously completed cells:')
        for k in sorted(self.state, key=lambda x: int(x)):
            print(f'   cell {k}: {self.state[k]}')
        nxt = max(int(k) for k in self.state) + 1
        print(f'-> After a disconnect, re-run cells 1..{nxt-1} (they are cached '
              f'and fast), then continue at cell {nxt}.')


---
## Cell 1 — Setup

Identical parameters to notebook 01. Re-reads the stage-1 cache from Drive, so nothing is recollected.

In [ ]:
# ============================ CELL 1 — SETUP ============================
!pip -q install rdrobust pandas numpy scipy matplotlib requests pyarrow tqdm statsmodels 2>/dev/null

if 'ResumableCollector' not in globals():
    raise SystemExit('>>> Run CELL 0 first (crash-safe infrastructure).')

import os, json, time, math, re, subprocess, urllib.parse, urllib.request
import datetime as dt
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from rdrobust import rdrobust

try:
    from google.colab import drive; drive.mount('/content/drive')
    WORK = Path('/content/drive/MyDrive/msr2027')
except Exception:
    WORK = Path('./msr2027')
CACHE = WORK/'cache'; CACHE.mkdir(parents=True, exist_ok=True)

ANNOUNCE_DATE    = dt.date(2022, 7, 8)
COHORT_SIZE      = 3800
COHORT_GRID      = [3500, 3600, 3700, 3800, 3900, 4000]
WAREHOUSE_COMMIT = '542f745f2ce03e01332516e9a20c2f2ae6b35742'
CONTACT_EMAIL    = 'anonymous.artifact@example.com'
T0               = pd.Timestamp(ANNOUNCE_DATE)
POST_DAYS        = 365

_y,_m = ANNOUNCE_DATE.year, ANNOUNCE_DATE.month-6
while _m<=0: _m+=12; _y-=1
WIN_START, WIN_END = dt.date(_y,_m,1), ANNOUNCE_DATE

rank = pd.read_parquet(CACHE/f'ranking_{WIN_START}_{WIN_END}.parquet')
CUT_DL = int(rank.loc[rank['rank']==COHORT_SIZE,'num_downloads'].iloc[0])

def rd(y, x, c=0.0, **kw):
    """rdrobust wrapper -> (conventional coef, robust p, effective n)."""
    y = np.asarray(y, float); x = np.asarray(x, float)
    ok = np.isfinite(y) & np.isfinite(x); y, x = y[ok], x[ok]
    try:
        r = rdrobust(y=y, x=x, c=c, **kw)
        try:    n = int(np.sum(np.asarray(r.N_h)))
        except Exception: n = len(y)
        return float(r.coef.iloc[0,0]), float(r.pv.iloc[2,0]), n
    except Exception as e:
        return float('nan'), float('nan'), 0

STATE = CellState(CACHE/'checkpoint_stage2.json')
STATE.report()
STATE.mark(1)
SETUP_DONE = True
print(f'CELL 1 OK — ranking {len(rank):,} projects, cutoff {CUT_DL:,} downloads')

---
## Cell 2 — Diagnose the outcome

Before choosing anything, look at the data. This cell prints the distribution of `n_rel_post`, names the extreme projects, and re-estimates the stage-1 effect under several outcome transformations.

What to expect: a long right tail, a handful of automated-release projects at the top, and an estimate whose magnitude collapses once the tail is controlled. If the raw-count estimate is an artefact, `log1p` and winsorised specifications will disagree with it sharply. If it survives all of them, the effect is real and the stage-1 estimate was merely imprecise.

In [ ]:
# ==================== CELL 2 — OUTCOME DIAGNOSIS ========================
if not globals().get('SETUP_DONE'): raise SystemExit('>>> Run CELL 1 first.')

cands = sorted(CACHE.glob('outcomes_*.parquet'))
if not cands: raise SystemExit('>>> No stage-1 outcomes found. Re-run notebook 01.')
df = pd.read_parquet(cands[-1]); print(f'Loaded {cands[-1].name}: {len(df):,} projects')

y = df['n_rel_post']
print('\n--- DISTRIBUTION OF n_rel_post (releases in 12 months post) ---')
print(y.describe(percentiles=[.5,.75,.9,.95,.99]).to_string())
print(f'\nshare with zero releases : {(y==0).mean():.1%}')
print(f'share above 50 releases  : {(y>50).mean():.2%}')
print(f'top 1% accounts for      : {y.nlargest(max(1,len(y)//100)).sum()/y.sum():.1%} of all releases')

print('\n--- 15 MOST EXTREME PROJECTS ---')
print(df.nlargest(15,'n_rel_post')[['project','rank','n_rel_post','n_rel_pre','treated']]
        .to_string(index=False))

print('\n--- ESTIMATE UNDER DIFFERENT SPECIFICATIONS ---')
q99 = y.quantile(0.99)
specs = {
 'raw count (stage 1)'      : y.astype(float),
 'winsorised at p99'      : y.clip(upper=q99).astype(float),
 'log1p(releases)'        : np.log1p(y),
 'released at all (0/1)'  : (y>0).astype(float),
 'released >=3 times (0/1)': (y>=3).astype(float),
 'days to next release'   : df['days_to_next'].astype(float),
}
rows=[]
for name, yy in specs.items():
    est,pv,n = rd(yy, df['running'])
    rows.append(dict(spec=name, estimate=est, robust_p=pv, n_eff=n))
res = pd.DataFrame(rows)
print(res.to_string(index=False, float_format=lambda v: f'{v:9.4f}'))

fig,ax=plt.subplots(1,2,figsize=(13,3.8))
ax[0].hist(y, bins=60, color='#4C78A8'); ax[0].set_yscale('log')
ax[0].set_title('n_rel_post — raw (log y)'); ax[0].set_xlabel('releases')
ax[1].hist(np.log1p(y), bins=40, color='#59A14F')
ax[1].set_title('log1p(n_rel_post)'); ax[1].set_xlabel('log1p(releases)')
plt.tight_layout(); plt.show()
STATE.mark(2)
DIAG_DONE = True; print('\nCELL 2 OK')

---
## Cell 3 — Fix the primary outcome, before seeing its estimate

The order matters. Choosing an outcome *after* seeing which one gives the nicest p-value is specification search, and it is the thing a sceptical reviewer will most want to accuse you of. So the choice is made here on stated grounds, written down, and not revisited.

**Primary: `released_at_all` — did the project ship any release in the 12 months after designation.** Bounded in [0,1], immune to the automated-release tail, and it maps directly onto the question the paper actually asks: did a mandated security control push maintainers to stop maintaining. It is also the outcome a policy audience understands without a footnote.

**Secondary: `log1p(releases)`** for intensity, and **`days_to_next_release`** as a survival-flavoured check. Raw counts are reported in an appendix for transparency, flagged as tail-sensitive.

This cell writes the choice to a timestamped file in Drive. That file is your evidence of pre-specification if anyone asks — a cheap, credible substitute for full OSF pre-registration under submission time constraints.

In [ ]:
# =================== CELL 3 — PRE-SPECIFICATION =========================
if not globals().get('DIAG_DONE'): raise SystemExit('>>> Run CELL 2 first.')

SPEC = {
 'decided_at'      : dt.datetime.utcnow().isoformat()+'Z',
 'primary_outcome' : 'released_at_all  = 1[releases in (T0, T0+365d] > 0]',
 'primary_rationale': ('bounded in [0,1]; immune to the automated-release tail that '
                       'made raw counts implausible in stage 1; directly answers whether '
                       'mandated 2FA pushed maintainers to stop maintaining'),
 'secondary'       : ['log1p(releases_post)', 'days_to_next_release (censored at 365)'],
 'appendix_only'   : ['raw release count (tail-sensitive)'],
 'running_variable': 'log(downloads_in_PyPI_window) - log(downloads_at_rank_COHORT_SIZE)',
 'cutoff'          : 0.0,
 'bandwidth'       : 'rdrobust MSE-optimal (mserd), data-driven; sensitivity over 0.5x-2x',
 'inference'       : 'robust bias-corrected (Calonico-Cattaneo-Titiunik)',
 'cohort_size'     : COHORT_SIZE,
 'cohort_sensitivity': COHORT_GRID,
 'estimand'        : ('INTENT-TO-TREAT: effect of DESIGNATION, not of enabling 2FA. '
                      'Per-account compliance is not public.'),
 'falsification'   : ['placebo cutoffs', 'donut-hole', 'covariate balance',
                      'placebo pre-period date'],
}
p = CACHE/f'PRESPEC_{dt.date.today()}.json'
p.write_text(json.dumps(SPEC, indent=2))
print(json.dumps(SPEC, indent=2))
print(f'\nWritten to {p}')
STATE.mark(3)
SPEC_DONE = True; print('\nCELL 3 OK')

---
## Cell 4 — Full outcome collection

Stage 1 sampled 1,500 projects for speed. Now collect the entire band.

Checkpointed every 500 projects to Drive, so a Colab disconnect costs you minutes rather than the run. Re-running resumes where it stopped. At roughly 12 workers this is on the order of 10–25 minutes for a band of a few thousand.

In [ ]:
# ================== CELL 4 — FULL COLLECTION (RESUMABLE) ================
if not globals().get('SPEC_DONE'): raise SystemExit('>>> Run CELL 3 first.')

# Rebuild the analysis band (cheap, deterministic — no network).
rank['norm'] = rank['project'].str.lower().str.replace('_','-', regex=False)
if not Path('/content/wh').exists():
    subprocess.run(['git','clone','-q','--filter=blob:none',
                    'https://github.com/pypi/warehouse.git','/content/wh'])
subprocess.run(['git','-C','/content/wh','checkout','-q',WAREHOUSE_COMMIT])
always=set()
for f in ['requirements/main.txt','requirements/deploy.txt']:
    fp=Path('/content/wh')/f
    if fp.exists():
        for line in fp.read_text().splitlines():
            line=line.split('#')[0].strip()
            if not line or line.startswith('-'): continue
            n=re.split(r'[=<>!\[;~]', line)[0].strip()
            if n: always.add(n.lower().replace('_','-'))
rank['always_taker']=rank['norm'].isin(always)
rank_clean = rank[~((rank['rank']>COHORT_SIZE)&rank['always_taker'])].copy()

band = rank_clean[(rank_clean.num_downloads>CUT_DL*0.5)&
                  (rank_clean.num_downloads<CUT_DL*2.0)].copy()
band['treated']=(band['rank']<=COHORT_SIZE).astype(int)
band['running']=np.log(band.num_downloads)-np.log(CUT_DL)
print(f'Full band: {len(band):,} projects '
      f'({band.treated.sum():,} treated / {(1-band.treated).sum():,} control)\n')

sess = requests.Session()
sess.headers.update({'User-Agent':f'MSR2027-research ({CONTACT_EMAIL})'})

def upload_times(pr):
    try:
        r=sess.get(f'https://pypi.org/pypi/{pr}/json', timeout=25)
        if r.status_code==200:
            rel=r.json().get('releases')
            if rel:
                ts=[pd.Timestamp(f['upload_time_iso_8601']).tz_localize(None)
                    for v in rel.values() for f in v[:1] if f.get('upload_time_iso_8601')]
                if ts: return sorted(ts)
        elif r.status_code==404:
            return 'GONE'
    except Exception: pass
    try:
        r=sess.get(f'https://pypi.org/simple/{pr}/', timeout=25,
                   headers={'Accept':'application/vnd.pypi.simple.v1+json'})
        if r.status_code==200:
            ts=[pd.Timestamp(f['upload-time']).tz_localize(None)
                for f in r.json().get('files',[]) if f.get('upload-time')]
            if ts: return sorted(ts)
    except Exception: pass
    return None

def rec(pr):
    ts=upload_times(pr)
    if ts=='GONE':      # removed from PyPI: a real observation, not a failure
        return dict(age_days=None, n_rel_pre=None, n_rel_post=None,
                    days_to_next=None, abandoned=None, removed=1)
    if not ts: return None
    s=pd.Series(ts); post=s[(s>=T0)&(s<T0+pd.Timedelta(days=POST_DAYS))]
    return dict(age_days=(T0-s.iloc[0]).days,
        n_rel_pre=int(((s>=T0-pd.Timedelta(days=POST_DAYS))&(s<T0)).sum()),
        n_rel_post=int(len(post)),
        days_to_next=(post.iloc[0]-T0).days if len(post) else POST_DAYS,
        abandoned=int(s.iloc[-1] < T0+pd.Timedelta(days=730)), removed=0)

# --- Crash-safe, resumable. Interrupt or disconnect any time; re-run to resume.
COL = ResumableCollector('pypi_outcomes',
                         local_dir='/content/msr_state',
                         mirror_dir=str(CACHE/'state'),
                         workers=12, flush_every=10,
                         mirror_every=100, mirror_seconds=60)
got = COL.run(list(band['project']), rec)

full = band.merge(got.drop(columns=[c for c in ['removed'] if c not in got.columns],
                           errors='ignore'), on='project')
full = full[full['n_rel_post'].notna()].copy()
HIT = len(full)/len(band)
print(f'\nUsable rows {len(full):,} / band {len(band):,}  (coverage {HIT:.1%})')
print(f'Removed-from-PyPI observations: {int(got["removed"].sum()) if "removed" in got else 0}')
STATE.mark(4); FULL_DONE = True; print('\nCELL 4 OK')

---
## Cell 5 — Primary result and robustness

Everything here was fixed in cell 3. Nothing below changes the specification.

The falsification tests are the point of this cell, not the headline number. Placebo cutoffs ask whether a "discontinuity" appears at arbitrary thresholds where no policy exists — if it does, yours is an artefact of functional form. The donut-hole drops observations immediately at the cutoff, where the partial-final-month approximation makes treatment status least certain. Bandwidth sensitivity asks whether the result depends on a single algorithmic choice.

In [ ]:
# ============ CELL 5 — PRIMARY RESULT + ROBUSTNESS ======================
if not globals().get('FULL_DONE'): raise SystemExit('>>> Run CELL 4 first.')

full['released_at_all'] = (full['n_rel_post']>0).astype(float)
full['log1p_rel']       = np.log1p(full['n_rel_post'])

print('='*70); print('PRIMARY: released_at_all'); print('='*70)
EST,PV,NEFF = rd(full['released_at_all'], full['running'])
base = full.loc[full['running']<0,'released_at_all'].mean()
print(f'  effect = {EST:+.4f} pp   robust p = {PV:.4f}   n_eff = {NEFF}')
print(f'  control-side base rate = {base:.3f}  -> relative change {EST/base:+.1%}' if base else '')

print('\n'+'='*70); print('SECONDARY'); print('='*70)
for nm,yy in [('log1p(releases)', full['log1p_rel']),
              ('days to next release', full['days_to_next'].astype(float))]:
    e,p_,n = rd(yy, full['running']); print(f'  {nm:24s} {e:+10.4f}  p={p_:.4f}  n={n}')

print('\n'+'='*70); print('FALSIFICATION 1 — placebo cutoffs (expect NULL)'); print('='*70)
hits=0; tot=0
for pr in [1500,2000,2500,3000,5000,6000,7000,8000]:
    sel = rank_clean.loc[rank_clean['rank']==pr,'num_downloads']
    if not len(sel): continue
    c2=float(sel.iloc[0])
    dd=full.assign(r2=np.log(full.num_downloads)-np.log(c2))
    dd=dd[(dd.r2>-0.7)&(dd.r2<0.7)]
    if len(dd)<150: continue
    e,p_,n = rd(dd['released_at_all'], dd['r2']); tot+=1; hits += int(p_<0.05)
    print(f'  rank {pr:5d}: {e:+8.4f}  p={p_:.3f}  n={n}   {"<-- SPURIOUS" if p_<0.05 else ""}')
print(f'  {hits}/{tot} placebo cutoffs significant at 5% (expect ~{0.05*tot:.1f})')

print('\n'+'='*70); print('FALSIFICATION 2 — donut-hole'); print('='*70)
for d in [0.0,0.02,0.05,0.10]:
    dd=full[np.abs(full['running'])>=d]
    e,p_,n = rd(dd['released_at_all'], dd['running'])
    print(f'  donut {d:.2f}: {e:+8.4f}  p={p_:.3f}  n={n}')

print('\n'+'='*70); print('FALSIFICATION 3 — bandwidth sensitivity'); print('='*70)
for h in [0.15,0.25,0.35,0.50,0.70]:
    dd=full[np.abs(full['running'])<=h]
    e,p_,n = rd(dd['released_at_all'], dd['running'], h=h)
    print(f'  h={h:.2f}: {e:+8.4f}  p={p_:.3f}  n={n}')

print('\n'+'='*70); print('COHORT-SIZE SENSITIVITY'); print('='*70)
for cs in COHORT_GRID:
    sel=rank_clean.loc[rank_clean['rank']==cs,'num_downloads']
    if not len(sel): continue
    c2=float(sel.iloc[0])
    dd=full.assign(r2=np.log(full.num_downloads)-np.log(c2))
    dd=dd[(dd.r2>-0.7)&(dd.r2<0.7)]
    e,p_,n = rd(dd['released_at_all'], dd['r2'])
    print(f'  cohort={cs}: {e:+8.4f}  p={p_:.3f}  n={n}')
STATE.mark(5)
ROBUST_DONE = True; print('\nCELL 5 OK')

---
## Cell 6 — Minimum detectable effect

**Do not skip this, and do not skip it especially if the result is null.**

A null with adequate power is a real, citable finding: mandated designation did not measurably push maintainers off their projects, which is exactly what PyPI, npm and the CRA drafters have been assuming without evidence. A null without power is nothing at all, and presenting one as the other is the single most common way this genre of paper fails review.

This cell computes what effect size your design could have detected at 80% power. Report it in the abstract if the headline is a null.

In [ ]:
# ==================== CELL 6 — POWER / MDE ==============================
if not globals().get('ROBUST_DONE'): raise SystemExit('>>> Run CELL 5 first.')
from scipy import stats

r = rdrobust(y=full['released_at_all'].values.astype(float),
             x=full['running'].values.astype(float), c=0)
se = float(r.se.iloc[2,0]); n_eff = int(np.sum(np.asarray(r.N_h)))
MDE = (stats.norm.ppf(0.975)+stats.norm.ppf(0.80))*se
base = full.loc[full['running']<0,'released_at_all'].mean()

print(f'  robust SE                : {se:.5f}')
print(f'  effective n              : {n_eff}')
print(f'  MDE at 80% power, a=0.05 : {MDE:.4f} ({MDE/base:+.1%} of the {base:.3f} base rate)')
print(f'  observed estimate        : {EST:+.4f}  (p={PV:.4f})')
print()
if PV >= 0.05:
    print(f'  NULL RESULT. Powered to detect effects of {MDE/base:.1%} or larger.')
    print(f'  Frame as: "designation did not reduce continuation by more than')
    print(f'  {MDE/base:.0%}; we can rule out effects larger than that."')
    print(f'  That is a finding. State the MDE in the abstract.')
else:
    print(f'  SIGNIFICANT at 5%. Confirm it survives every falsification test in cell 5')
    print(f'  before believing it. A result that appears at placebo cutoffs is not real.')
STATE.mark(6)
POWER_DONE = True; print('\nCELL 6 OK')

---
## Cell 7 — Export and handoff


In [ ]:
# ==================== CELL 7 — EXPORT ===================================
if not globals().get('POWER_DONE'): raise SystemExit('>>> Run CELL 6 first.')
out = {
 'run_at': dt.datetime.utcnow().isoformat()+'Z',
 'n_band': int(len(band)), 'n_collected': int(len(full)), 'coverage': float(HIT),
 'n_treated': int(full.treated.sum()), 'n_control': int((1-full.treated).sum()),
 'cutoff_downloads': int(CUT_DL), 'cohort_size': COHORT_SIZE,
 'primary_estimate': float(EST), 'primary_robust_p': float(PV),
 'primary_n_eff': int(NEFF), 'control_base_rate': float(base),
 'mde_80pct': float(MDE), 'robust_se': float(se),
}
(CACHE/'results_rdd.json').write_text(json.dumps(out, indent=2))
full.to_parquet(CACHE/'analysis_frame_rdd.parquet')
print(json.dumps(out, indent=2))
print('\nSaved results_rdd.json + analysis_frame_rdd.parquet')